<a href="https://colab.research.google.com/github/temraire117/NLP/blob/main/NLP_week04_%EB%B0%B0%ED%8F%AC%EC%9A%A9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 4주차 실습: 한글 형태소 분석과 토큰화

**주제:** Kiwi·KoNLPy(Okt)를 이용한 형태소 분석, 품사 태깅, 사용자 사전

## 학습 목표
1. 어절·형태소·토큰의 차이를 설명한다.
2. 공백 기반 토큰화와 형태소 분석 기반 토큰화를 비교한다.
3. Kiwi와 Okt로 품사 태깅을 수행하고 과업에 맞는 분석기를 판단한다.
4. 사용자 사전이 필요한 사례를 만들고 적용한다.

> **제출:** 모든 TODO를 작성·실행하고, 체크포인트 답변을 Markdown 셀에 남기세요. 외부 AI·자료를 사용했다면 사용 범위와 검증 과정을 기록하세요.

## 0. 준비
교육용 예시 데이터입니다. 실제 데이터는 출처·라이선스·개인정보 포함 여부를 확인하세요.

In [2]:
# 실행 오류가 날 때만 아래 주석을 해제하세요.
!pip -q install -U kiwipiepy konlpy

import re
import pandas as pd
from kiwipiepy import Kiwi
from konlpy.tag import Okt

kiwi = Kiwi()
okt = Okt()

texts = [
    '충북대학교 자연어처리 수업이 정말 재미있어요!',
    'AI 챗봇의 답변 품질을 개선하고 싶어요.',
    '배터리가 오래가고 화면도 선명해서 만족합니다.',
    '결말은 아쉬웠지만 배우 연기는 최고였어요.',
    '새로 나온 갓생 챌린지 앱을 사용해 봤어요.'
]
df = pd.DataFrame({'text_id': range(1, len(texts)+1), 'text': texts})
display(df)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 MB 9.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 69.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.5/438.5 kB 25.0 MB/s eta 0:00:00


,text_id,text
0,1,충북대학교 자연어처리 수업이 정말 재미있어요!
1,2,AI 챗봇의 답변 품질을 개선하고 싶어요.
2,3,배터리가 오래가고 화면도 선명해서 만족합니다.
3,4,결말은 아쉬웠지만 배우 연기는 최고였어요.
4,5,새로 나온 갓생 챌린지 앱을 사용해 봤어요.


## 1. 어절·형태소·토큰
- **어절:** 띄어쓰기로 구분되는 단위
- **형태소:** 뜻 또는 문법 기능을 지닌 최소 단위
- **토큰:** 분석 목적에 맞게 모델 입력으로 정한 단위

예: `학생들이 도서관에서 공부한다.` → `학생/들/이`, `도서관/에서`, `공부/하/ㄴ다`처럼 분해할 수 있습니다.

In [4]:
sample = '학생들이 도서관에서 공부한다.'
print('원문:', sample)
print('공백 토큰:', sample.split())
print('정규표현식 토큰:', re.findall(r'[가-힣A-Za-z0-9]+', sample))

# TODO 1: Kiwi 형태소 분석 결과를 [(형태, 품사), ...] 형태로 출력하세요.
kiwi_result = [(token.form,token.tag) for token in kiwi.tokenize(sample)]
print(kiwi_result)

원문: 학생들이 도서관에서 공부한다.
공백 토큰: ['학생들이', '도서관에서', '공부한다.']
정규표현식 토큰: ['학생들이', '도서관에서', '공부한다']
[('학생', 'NNG'), ('들', 'XSN'), ('이', 'JKS'), ('도서관', 'NNG'), ('에서', 'JKB'), ('공부', 'NNG'), ('하', 'XSV'), ('ᆫ다', 'EF'), ('.', 'SF')]


### 체크포인트 1
공백 토큰화로는 보이지 않는 문법·의미 정보 한 가지를 위 문장에서 찾아 1~2문장으로 설명하세요.

- 공백 토큰화에서는 '학생들이'가 하나의 단어로 처리되지만, 형태소 분석을 통해 '학생(NNG)', '들(XSN)', '이(JKS)'로 분리할 수 있다. 이를 통해 '들'은 복수의 의미를, '이'는 주어를 나타내는 문법적 기능을 한다는 점을 알 수 있다.

## 2. Kiwi로 형태소 분석과 품사 태깅
Kiwi는 현대 한국어 분석에 활용하기 좋은 형태소 분석기입니다. `token.form`은 형태, `token.tag`는 품사 태그입니다.

In [5]:
def kiwi_pos(text):
    return [(token.form, token.tag) for token in kiwi.tokenize(text)]

for text in df['text']:
    print(text)
    print(kiwi_pos(text))
    print()

# TODO 2: 아래 문장의 명사(태그가 N으로 시작)만 추출하는 함수를 작성하세요.
def kiwi_nouns(text):
    return [token.form for token in kiwi.tokenize(text)]
    pass

print(kiwi_nouns(df.loc[0, 'text']))

충북대학교 자연어처리 수업이 정말 재미있어요!
[('충북대학교', 'NNP'), ('자연어 처리', 'NNP'), ('수업', 'NNG'), ('이', 'JKS'), ('정말', 'MAG'), ('재미있', 'VA'), ('어요', 'EF'), ('!', 'SF')]

AI 챗봇의 답변 품질을 개선하고 싶어요.
[('AI', 'SL'), ('챗봇', 'NNG'), ('의', 'JKG'), ('답변', 'NNG'), ('품질', 'NNG'), ('을', 'JKO'), ('개선', 'NNG'), ('하', 'XSV'), ('고', 'EC'), ('싶', 'VX'), ('어요', 'EF'), ('.', 'SF')]

배터리가 오래가고 화면도 선명해서 만족합니다.
[('배터리', 'NNG'), ('가', 'JKS'), ('오래', 'MAG'), ('가', 'VV'), ('고', 'EC'), ('화면', 'NNG'), ('도', 'JX'), ('선명', 'NNG'), ('하', 'XSA'), ('어서', 'EC'), ('만족', 'NNG'), ('하', 'XSA'), ('ᆸ니다', 'EF'), ('.', 'SF')]

결말은 아쉬웠지만 배우 연기는 최고였어요.
[('결말', 'NNG'), ('은', 'JX'), ('아쉽', 'VA-I'), ('었', 'EP'), ('지만', 'EC'), ('배우', 'NNG'), ('연기', 'NNG'), ('는', 'JX'), ('최고', 'NNG'), ('이', 'VCP'), ('었', 'EP'), ('어요', 'EF'), ('.', 'SF')]

새로 나온 갓생 챌린지 앱을 사용해 봤어요.
[('새로', 'MAG'), ('나오', 'VV'), ('ᆫ', 'ETM'), ('갓', 'MAG'), ('생', 'XPN'), ('챌린지', 'NNG'), ('앱', 'NNG'), ('을', 'JKO'), ('사용', 'NNG'), ('하', 'XSV'), ('어', 'EC'), ('보', 'VX'), ('었', 'EP'), ('어요', 'E

## 3. KoNLPy의 Okt로 형태소 분석
Okt는 소셜 텍스트·이모티콘·비표준 표현을 살펴볼 때 자주 사용합니다. 분석기마다 분리 방식과 품사 체계가 다를 수 있습니다.

In [7]:
compare_text = '오늘 수업 진짜 재밌었는데 과제는 조금 어렵네요 ㅠㅠ'
print('원문:', compare_text)
print('Kiwi:', kiwi_pos(compare_text))
print('Okt :', okt.pos(compare_text))

# TODO 3: 아래 문장을 Kiwi와 Okt로 각각 분석하고 결과 차이를 관찰하세요.
my_sentence = '신조어와 줄임말도 분석할 수 있을까?'
print('Kiwi:', kiwi_pos(my_sentence))
print('Okt :', okt.pos(my_sentence))

원문: 오늘 수업 진짜 재밌었는데 과제는 조금 어렵네요 ㅠㅠ
Kiwi: [('오늘', 'NNG'), ('수업', 'NNG'), ('진짜', 'MAG'), ('재밌', 'VA'), ('었', 'EP'), ('는데', 'EC'), ('과제', 'NNG'), ('는', 'JX'), ('조금', 'MAG'), ('어렵', 'VA-I'), ('네요', 'EF'), ('ㅠㅠ', 'SW')]
Okt : [('오늘', 'Noun'), ('수업', 'Noun'), ('진짜', 'Noun'), ('재밌었는데', 'Adjective'), ('과제', 'Noun'), ('는', 'Josa'), ('조금', 'Noun'), ('어렵네요', 'Adjective'), ('ㅠㅠ', 'KoreanParticle')]
Kiwi: [('신조어', 'NNG'), ('와', 'JC'), ('줄임', 'NNG'), ('말', 'NNG'), ('도', 'JX'), ('분석', 'NNG'), ('하', 'XSV'), ('ᆯ', 'ETM'), ('수', 'NNB'), ('있', 'VA'), ('을까', 'EF'), ('?', 'SF')]
Okt : [('신조어', 'Noun'), ('와', 'Josa'), ('줄임말도', 'Verb'), ('분석', 'Noun'), ('할', 'Verb'), ('수', 'Noun'), ('있을까', 'Adjective'), ('?', 'Punctuation')]


### 체크포인트 2
Kiwi와 Okt 결과에서 토큰 분리 또는 품사 태깅이 다른 사례를 하나 기록하세요. 어느 분석기가 “더 좋다”가 아니라, 어떤 **데이터/과업**에서 더 편리할지 설명하세요.

- '재밌었는데'를 Kiwi는 '재밌', '었', '는데' 로, Okt는 '재밌었는데'로 태깅을 했다. Kiwi는 연결 어미 등의 문법 정보를 분석하기 편리하고, Okt는 자연스러운 문장 분석에 편리할 것이다.

## 4. 과업에 따른 품사 선택
키워드 추출은 명사 중심 처리가 유용할 수 있습니다. 반면 감성 분석에서는 형용사·동사·부정 표현·강조 부사도 중요한 신호가 됩니다.

In [8]:
review = '배송은 조금 느렸지만 제품 디자인이 매우 예쁘고 만족스러워요.'
print('전체 분석:', kiwi_pos(review))

# TODO 4: 명사(N*)와 용언(V*)을 각각 추출하세요.
def select_pos(text):
    nouns = []
    predicates = []
    for token in kiwi.tokenize(text):
      nouns.append(token.form) if token.tag.startswith('N') else None
      predicates.append(token.form) if token.tag.startswith('V') else None
    return nouns, predicates

nouns, predicates = select_pos(review)
print('명사:', nouns)
print('용언:', predicates)

전체 분석: [('배송', 'NNG'), ('은', 'JX'), ('조금', 'MAG'), ('느리', 'VA'), ('었', 'EP'), ('지만', 'EC'), ('제품', 'NNG'), ('디자인', 'NNG'), ('이', 'JKS'), ('매우', 'MAG'), ('예쁘', 'VA'), ('고', 'EC'), ('만족', 'NNG'), ('스럽', 'XSA-I'), ('어요', 'EF'), ('.', 'SF')]
명사: ['배송', '제품', '디자인', '만족']
용언: ['느리', '예쁘']


### 체크포인트 3
`좋지 않아요`, `매우 예쁘고` 같은 표현을 생각해 보세요. 명사만 남기면 감성 분석에서 어떤 정보가 사라질까요?

- '좋지 않아요'를 생각해 볼 경우 부사와 동사로 이루어져 있으며, 이것을 삭제하면 긍정/부정 정보가 사라질 수 있다.

## 5. 사용자 사전: 왜 필요한가
새 서비스명, 과목명, 브랜드명, 전문 용어는 일반 사전에 없거나 원하지 않는 방식으로 분리될 수 있습니다. 사용자 사전은 **도메인 용어를 원하는 품사와 단위로 분석**하도록 돕습니다.

아래 코드는 Kiwi 버전에 따라 사용자 단어 추가 방식이 다를 수 있으므로, 실행 환경의 오류 메시지와 라이브러리 문서를 함께 확인하세요.

In [9]:
domain_text = '충북대학교에서 자연어처리, 머신러닝, 딥러닝, 프로젝트를 진행한다.'
print('추가 전:', kiwi_pos(domain_text))

# TODO 5: 사용자 단어를 추가한 뒤 다시 분석하세요.
# 예시: kiwi.add_user_word('자연어처리', 'NNP')
kiwi.add_user_word('충북대학교', 'NNP')
kiwi.add_user_word('머신러닝', 'NNP')
kiwi.add_user_word('딥러닝', 'NNP')
print('추가 후:', kiwi_pos(domain_text))

# TODO 6: 본인의 전공/관심 분야 고유명사 2개를 추가하고, 전후 결과를 비교하세요.

추가 전: [('충북대학교', 'NNP'), ('에서', 'JKB'), ('자연어 처리', 'NNP'), (',', 'SP'), ('머신', 'NNG'), ('러닝', 'NNG'), (',', 'SP'), ('딥', 'NNG'), ('러닝', 'NNG'), (',', 'SP'), ('프로젝트', 'NNG'), ('를', 'JKO'), ('진행', 'NNG'), ('하', 'XSV'), ('ᆫ다', 'EF'), ('.', 'SF')]
추가 후: [('충북대학교', 'NNP'), ('에서', 'JKB'), ('자연어 처리', 'NNP'), (',', 'SP'), ('머신러닝', 'NNP'), (',', 'SP'), ('딥러닝', 'NNP'), (',', 'SP'), ('프로젝트', 'NNG'), ('를', 'JKO'), ('진행', 'NNG'), ('하', 'XSV'), ('ᆫ다', 'EF'), ('.', 'SF')]


## 6. 미니 과제: 토큰화 정책 설계
다음 목표 중 하나를 선택해 토큰화 정책을 설계하세요.
1. 강의평 키워드 추출
2. 영화 리뷰 감성 분석
3. 학교 민원 자동 분류

아래 함수에 분석기, 보존할 품사, 제거할 토큰의 기준을 작성하세요.

In [10]:
def tokenize_for_task(text):
    """TODO 7: 선택한 과업에 맞는 토큰화 규칙을 구현하세요.
    - 어떤 품사를 남길지: 일반명사(NNG), 고유명사(NNP), 동사(V), 형용사(VA), 부정 관련 부사 및 부정 표현
    - 조사/기호/URL 등을 어떻게 다룰지: 주격, 목적격조사는 제거, 기호 및 URL 또한 제거
    - 부정 표현·이모지·반복 문자를 보존할지: 감정 분석을 위해 보존
    """
    tokens = []
    # TODO
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)

    text = re.sub(r'(ㅋ)\1{2,}', r'\1\1', text)
    text = re.sub(r'(ㅠ)\1{2,}', r'\1\1', text)

    analyzed = kiwi.tokenize(text)

    for token in analyzed:
        word = token.form
        tag = token.tag

        if (tag.startswith('N') or
            tag.startswith('V') or
            word in ['안', '못', '않다', '없다', '아니다']):
            tokens.append(word)

        elif re.fullmatch(r'[ㅋㅠㅜㅎ]+', word):
            tokens.append(word)

    return tokens

for text in df['text']:
    print(text, '->', tokenize_for_task(text))

충북대학교 자연어처리 수업이 정말 재미있어요! -> ['충북대학교', '자연어 처리', '수업', '재미있']
AI 챗봇의 답변 품질을 개선하고 싶어요. -> ['챗봇', '답변', '품질', '개선', '싶']
배터리가 오래가고 화면도 선명해서 만족합니다. -> ['배터리', '가', '화면', '선명', '만족']
결말은 아쉬웠지만 배우 연기는 최고였어요. -> ['결말', '아쉽', '배우', '연기', '최고', '이']
새로 나온 갓생 챌린지 앱을 사용해 봤어요. -> ['나오', '챌린지', '앱', '사용', '보']


### 체크포인트 4
선택한 과업, 분석기 선택 이유, 보존/제거 규칙, 예상되는 오류 사례를 각각 1문장 이상 작성하세요.

1. 선택한 과업

학교 민원 자동 분류를 선택하여 학생들의 민원을 시설 문제, 수업 문제, 행정 서비스 등으로 자동 분류하고자 한다.

2. 분석기 선택 이유

Kiwi 형태소 분석기는 명사뿐 아니라 동사, 형용사, 부정 표현 등을 형태소 단위로 분석함으로써 민원의 핵심 내용과 요구사항을 파악하기 용이하다.

3. 보존/제거 규칙

명사, 동사, 형용사와 '안', '못', '않다' 등의 부정 표현은 보존하고, 조사와 문장부호는 제거하며 URL은 마스킹한다. 이모지와 반복 문자는 민원의 감정이나 불만 강도를 파악하는 데 활용할 수 있도록 보존한다.

4. 예상되는 오류 사례

'수업이 나쁘지 않다'와 같은 문장에서 부정 표현이 제대로 보존되지 않으면 민원의 의미가 반대로 분류될 수 있다. 또한 '강의실이 엉망이다'와 같은 구어체나 비속어는 형태소 분석기가 정확하게 분석하지 못해 분류 오류가 발생할 수 있다.

**형성평가**

A1: 교착어

A2: 사용자 사전

Kiwi와 Okt의 결과 차이: Kiwi는 동사를 세분화하여 분석하는 데 특화되어있고, Okt는 동사를 묶어 비교적 간단히 분석하는 데 특화되어있다.

## 제출 체크리스트
- [ ] TODO 1~7을 작성하고 실행했다.
- [ ] 공백·정규표현식·형태소 기반 토큰화를 비교했다.
- [ ] Kiwi와 Okt의 결과 차이 및 과업 적합성을 설명했다.
- [ ] 명사 중심 처리의 장점과 한계를 설명했다.
- [ ] 사용자 사전에 고유명사 2개 이상을 추가하고 전후 결과를 비교했다.
- [ ] 선택 과업에 대한 토큰화 정책과 오류 사례를 기록했다.
- [ ] 외부 AI/자료 사용 시 범위와 검증 과정을 기록했다.

## 다음 주차 예고
전처리된 토큰을 Bag-of-Words, TF-IDF, n-gram으로 수치화하여 문서 비교와 분류 모델의 입력 특징을 구성합니다.